In [27]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Download dataset
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

csv_path = os.path.join(path, "Reviews.csv")



Using Colab cache for faster access to the 'amazon-fine-food-reviews' dataset.


In [28]:


# Load first 50,000 rows
df = pd.read_csv(csv_path, nrows=50000)



In [29]:
reviews = df["Text"].dropna()

# Reviews containing the word "light"
light_reviews = reviews[reviews.str.contains(r"\blight\b", case=False, regex=True)]

print("Total reviews containing 'light':", len(light_reviews))

Total reviews containing 'light': 1415


In [30]:
import re

food_reviews = []
illumination_reviews = []

illumination_keywords = [
    "box", "inside", "sunlight", "bright",
    "lamp", "window", "shine", "lighting",
    "package", "packaging"
]

for review in light_reviews:

    text = review.lower()

    # illumination context
    if any(word in text for word in illumination_keywords):
        illumination_reviews.append(review)

    # food/lightweight context
    elif any(word in text for word in [
        "taste","snack","oil","coffee","tea",
        "healthy","stomach","flavor","calorie",
        "cream","milk","diet","texture"
    ]):
        food_reviews.append(review)

print("Food reviews found:", len(food_reviews))
print("Illumination reviews found:", len(illumination_reviews))

Food reviews found: 958
Illumination reviews found: 288


In [31]:
if len(food_reviews) >= 2 and len(illumination_reviews) >= 1:

    sentences = [
        food_reviews[0],
        illumination_reviews[0],
        food_reviews[1]
    ]

else:
    print("No illumination-context review found in first 50,000 rows.")

    # fallback (only if necessary)
    sentences = [
        food_reviews[0],
        light_reviews.iloc[0],
        food_reviews[1]
    ]

In [32]:
print("="*100)
print("Sentence 1")
print(sentences[0])

print("="*100)
print("Sentence 2")
print(sentences[1])

print("="*100)
print("Sentence 3")
print(sentences[2])

Sentence 1
This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powdered sugar.  And it is a tiny mouthful of heaven.  Not too chewy, and very flavorful.  I highly recommend this yummy treat.  If you are familiar with the story of C.S. Lewis' "The Lion, The Witch, and The Wardrobe" - this is the treat that seduces Edmund into selling out his Brother and Sisters to the Witch.
Sentence 2
With Kettle Chips, you really have to be careful.  Some of their flavors are nauseating.  With that said, they DO make fantastic plain chips.  Thick cuts of potato, fried to a dark golden brown.  They are crunchy and lightly salted with sea salt.  I can't recommend these chips enough.  You won't regret it.<br /><br />Some people say they are burnt but they aren't.  From their website: "Take a quick look and you'll see an immediate difference: Kettle Brand® Potato 

In [33]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(sentences)

dense_matrix = X.toarray()

print("\nVocabulary:\n")
print(vectorizer.get_feature_names_out())

print("\nDense Matrix\n")
print(dense_matrix)


Vocabulary:

['20' 'acids' 'aftertaste' 'ahmad' 'also' 'am' 'amazon' 'amber' 'an' 'and'
 'any' 'are' 'area' 'aren' 'around' 'artisanal' 'as' 'at' 'available'
 'away' 'bag' 'be' 'beautiful' 'because' 'been' 'being' 'best' 'bitter'
 'both' 'br' 'brand' 'brother' 'brown' 'burnt' 'but' 'can' 'caramelize'
 'careful' 'carry' 'case' 'centuries' 'cheaper' 'chewy' 'chips' 'citrus'
 'coated' 'colors' 'confection' 'consistent' 'cooking' 'creating'
 'crunchy' 'current' 'cut' 'cuts' 'dark' 'deal' 'deep' 'delicate'
 'detected' 'difference' 'disappointed' 'display' 'do' 'drinking' 'during'
 'edmund' 'enough' 'even' 'every' 'exclusively' 'expeller' 'explain'
 'extra' 'familiar' 'fan' 'fantastic' 'fat' 'fats' 'fatty' 'few'
 'filberts' 'find' 'flavor' 'flavorful' 'flavors' 'floral' 'foods' 'for'
 'free' 'fried' 'from' 'gelatin' 'get' 'gold' 'golden' 'great' 'green'
 'happens' 'happy' 'has' 'have' 'heaven' 'here' 'high' 'highly' 'his'
 'hydrogenated' 'if' 'immediate' 'in' 'independent' 'indicate' 'intak

In [34]:
vec_1 = dense_matrix[0]
vec_2 = dense_matrix[1]
vec_3 = dense_matrix[2]

In [35]:
import numpy as np

score12 = np.dot(vec_1, vec_2)
score13 = np.dot(vec_1, vec_3)

print("\nSimilarity Scores")
print("-------------------------")
print("Sentence 1 vs Sentence 2 =", score12)
print("Sentence 1 vs Sentence 3 =", score13)


Similarity Scores
-------------------------
Sentence 1 vs Sentence 2 = 178
Sentence 1 vs Sentence 3 = 86


Question 1: The Vector Conflict

The similarity between Sentence 1 and Sentence 2 was 178, higher than 86. This happened because Bag-of-Words counts common words only, ignoring context, which creates a misleading similarity between different meanings of "light".

Question 2: The Context Blindspot

Bag-of-Words ignores context and word order, keeping only word frequencies. Because of this, search engines and chatbots may misunderstand words with multiple meanings and produce less accurate results.

Question 3: The GenAI Architectural Fix

Modern LLMs use Masked Self-Attention and Context-Aware Embeddings to understand surrounding words. This gives the word "light" different vector representations based on its meaning, improving language understanding and response accuracy.